# Build Clean Core Message Preprocessing Dataset

Purpose: create the clean message-level preprocessing dataset that feeds later threading, demographics/CCI, visit/encounter, and analytic merge notebooks.

This notebook prepares the message-level dataset through sender/source classification and stops before conversation-thread reconstruction.

## Preprocessing Scope

This notebook does:

- load and stack cleaned portal-message files
- normalize key message, subject, timestamp, and provider fields
- construct sender/receiver provider metadata
- create the final analytic sender/source variables used in the analysis: `Sender_all`, `Receiver_all`, and `origin`
- save the final pre-thread message-level handoff dataset as `messages_without_threads.parquet`
- write aggregate QA summaries and restricted output datasets to a timestamped run folder

This notebook does not:

- construct conversation threads; threading diagnostics and 15/30/60-day sensitivity analyses are handled in `02_threading_diagnostics_sensitivity.ipynb`
- analyze message body text for clinical content, symptoms, topics, sentiment, or reply intent
- merge demographics, Charlson CCI, visits, encounters, medications, or diagnoses
- fit statistical models or generate analysis figures
- print row-level patient or message examples

## Inputs

Inputs are configured in Section 1. The expected input folders are:

- cleaned message files, one file per message type
- provider/user mapping files used to classify sender and receiver user IDs

## Outputs

Outputs are written to a unique `runs/core_message_preprocessing_YYYYMMDD_HHMMSS/` folder. The canonical handoff output is also written as `messages_without_threads.parquet` in the configured working-output folder so downstream notebooks can read a stable file name.

## Privacy Constraints

The row-level outputs are derived from EHR portal-message records and should not be publicly shared. No row-level message text, patient identifiers, or patient examples should be printed in notebook output.

Processing principles:

- Source files are read-only.
- Timestamped run outputs are never overwritten.
- The stable `messages_without_threads.parquet` handoff file is a working derived dataset and may be refreshed when preprocessing is rerun.
- Notebook display/output should use aggregate QA only, not patient/message text examples.
- Comments explain analytic decisions and data handling choices.


## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import platform
import re
import sys
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd

pd.options.display.max_columns = 120


def find_repo_root() -> Path:
    """Locate the public repository root from the notebook or repository directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


def required_restricted_data_dir() -> Path:
    """Return the authorized local data root without embedding a machine-specific path."""
    value = os.environ.get("JMIR_RESTRICTED_DATA_DIR")
    if not value:
        raise EnvironmentError(
            "Set JMIR_RESTRICTED_DATA_DIR to the authorized local data root before running. "
            "See data/README.md for the expected directory layout."
        )
    return Path(value).expanduser().resolve()


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
RUNS_DIR = OUTPUT_ROOT / "runs"
WORKING_OUTPUT_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()

RESTRICTED_DATA_DIR = required_restricted_data_dir()
SOURCE_CLEANED_DIR = Path(
    os.environ.get("JMIR_CLEANED_MESSAGE_DIR", RESTRICTED_DATA_DIR / "Cleaned")
).expanduser().resolve()
PROVIDER_SOURCE_DIR = Path(
    os.environ.get("JMIR_PROVIDER_SOURCE_DIR", RESTRICTED_DATA_DIR / "Providers")
).expanduser().resolve()

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Stable handoff for the threading notebook; timestamped run files remain audit copies.
MESSAGES_WITHOUT_THREADS_CANONICAL_PATH = WORKING_OUTPUT_DIR / "messages_without_threads.parquet"
WRITE_CANONICAL_MESSAGES_WITHOUT_THREADS = os.environ.get(
    "JMIR_WRITE_CANONICAL_MESSAGES_WITHOUT_THREADS", "1"
) == "1"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / f"core_message_preprocessing_{RUN_ID}"
if RUN_DIR.exists():
    raise FileExistsError(f"Run directory already exists: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("REPO_ROOT:", REPO_ROOT)
print("RUN_DIR:", RUN_DIR)
print("SOURCE_CLEANED_DIR:", SOURCE_CLEANED_DIR)
print("PROVIDER_SOURCE_DIR:", PROVIDER_SOURCE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_OUTPUT_DIR:", WORKING_OUTPUT_DIR)
print("MESSAGES_WITHOUT_THREADS_CANONICAL_PATH:", MESSAGES_WITHOUT_THREADS_CANONICAL_PATH)

if not SOURCE_CLEANED_DIR.exists():
    raise FileNotFoundError(
        "SOURCE_CLEANED_DIR does not exist. Check JMIR_RESTRICTED_DATA_DIR or JMIR_CLEANED_MESSAGE_DIR."
    )
if not PROVIDER_SOURCE_DIR.exists():
    raise FileNotFoundError(
        "PROVIDER_SOURCE_DIR does not exist. Check JMIR_RESTRICTED_DATA_DIR or JMIR_PROVIDER_SOURCE_DIR."
    )

## 2. Helper Functions

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names without changing row values."""
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\u00ef\u00bb\u00bf", "", regex=False)
        .str.replace("\u00c3\u00af\u00c2\u00bb\u00c2\u00bf", "", regex=False)
        .str.strip()
    )
    return out


def read_message_file(path: Path, message_source: str) -> pd.DataFrame:
    """Read one cleaned message file and record its source file for QA."""
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, dtype=str, encoding="utf-8", encoding_errors="replace")
    df = clean_columns(df)
    df["message_source_file"] = message_source
    return df


def normalize_yes_no(series: pd.Series) -> pd.Series:
    """Convert Y/N flags to Yes/No while preserving unexpected source values."""
    yes_no_map = {"Y": "Yes", "N": "No"}
    normalized = series.astype("string").str.strip().str.upper().map(yes_no_map)
    return normalized.where(normalized.notna(), series.astype("string"))


def clean_subject_basic(text):
    """Remove reply/forward prefixes while preserving an interpretable fallback."""
    if pd.isna(text):
        return "Unknown"
    text = str(text)
    cleaned = re.sub(r"^((re|fw|fwd)\s*:\s*)+", "", text, flags=re.IGNORECASE)
    if cleaned.strip() == "" or re.fullmatch(r"((re|fw|fwd)\s*:\s*)+", text.strip(), flags=re.IGNORECASE):
        return "Unknown"
    return cleaned.strip()


def clean_topic_text(text):
    """Production subject normalization used for threading and message grouping."""
    if pd.isna(text):
        return "unknown"
    text = str(text).lower()
    # Keep hyphen, slash, and period because they commonly carry clinical or appointment meaning.
    text = re.sub(r"[^\w\s\-/\.]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text if text else "unknown"


TYPO_CORRECTIONS = {
    "apointment": "appointment",
    "appoinment": "appointment",
    "appointmet": "appointment",
    "appointmnt": "appointment",
    "appointmen": "appointment",
    "appointmetn": "appointment",
    "appointmnet": "appointment",
    "appointmn": "appointment",
    "apointmetn": "appointment",
    "apointmnt": "appointment",
    "apointmn": "appointment",
    "appoitment": "appointment",
    "apppointment": "appointment",
    "questionaire": "questionnaire",
    "questionaires": "questionnaires",
    "qustionnaire": "questionnaire",
    "questionnair": "questionnaire",
    "questionair": "questionnaire",
    "questionnairlist": "questionnairelist",
    "screning": "screening",
    "screenigns": "screenings",
    "covd": "covid",
    "coivd": "covid",
    "reasearch": "research",
    "procdure": "procedure",
}


def normalize_typos(text):
    if pd.isna(text):
        return text
    out = str(text).lower()
    for wrong, right in TYPO_CORRECTIONS.items():
        out = re.sub(rf"\b{wrong}\b", right, out)
    return out


SPANISH_SUBJECT_REPLACEMENTS = {
    "peticiÃ³n de cancelaciÃ³n de cita": "appointment cancellation request",
    "peticiÃ£Â³n de cancelaciÃ£Â³n de cita": "appointment cancellation request",
    "solicitud de cita": "appointment request",
    "confirmaciÃ³n de cita": "appointment confirmation",
    "confirmaciÃ£Â³n de cita": "appointment confirmation",
}


def normalize_known_subject_phrases(text):
    if pd.isna(text):
        return text
    out = str(text).lower()
    out = out.replace("\u00e2", "")
    out = out.replace("Ã¢", "")
    faulty = "covid 19 update from your pcp  please do not reply to message"
    fixed = "covid 19 update from your pcp please do not reply to message"
    if out.strip() == faulty:
        out = fixed
    for source, target in SPANISH_SUBJECT_REPLACEMENTS.items():
        out = out.replace(source, target)
    out = re.sub(r"\s+", " ", out).strip()
    return out if out else "unknown"


def qa_counts(df: pd.DataFrame, col: str) -> pd.DataFrame:
    out = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="count")
    )
    out["percent"] = (out["count"] / len(df) * 100).round(2)
    return out

## 3. Load And Stack Message Files

In [ ]:
message_files = [
    ("App_Cancel_Request_cleaned.csv", "Patient Appointment Cancel Request"),
    ("App_Schedule_Request_cleaned.csv", "Patient Appointment Schedule Request"),
    ("Case_Reminder_cleaned.csv", "Case Reminder Message"),
    ("Customer_Service_Request_cleaned.csv", "Patient Customer Service Request"),
    ("E_visit_cleaned.csv", "E-Visit"),
    ("Medical_Advice_Request_cleaned.csv", "Patient Medical Advice Request"),
    ("Medication_Renewal_Request_cleaned.csv", "Patient Medication Renewal Request"),
    ("User_Message_cleaned.csv", "User Message"),
    ("Referral_cleaned.csv", "Patient Referral Message"),
    ("System_Message_cleaned.csv", "System Message"),
    ("Treatment_Team_cleaned.csv", "Treatment Team Message"),
]

message_frames = []
source_file_rows = []
for filename, expected_type in message_files:
    path = SOURCE_CLEANED_DIR / filename
    df = read_message_file(path, filename)
    message_frames.append(df)
    source_file_rows.append({
        "source_file": filename,
        "expected_message_type": expected_type,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
    })

messages_raw = pd.concat(message_frames, ignore_index=True)
source_file_summary = pd.DataFrame(source_file_rows)

required_cols = [
    "MESSAGE_ID", "DMRN", "CREATED_TIME", "MESG_TYPE",
    "FROM_USER_ID", "TO_USER_ID", "TOFROM_PAT",
    "NOTALLOW_REPLY_YN", "REPLY_DIRECT_YN",
    "Prov_DEPT_For_Appt", "SUBJECT", "FINAL_HANDLED_TIME",
    "EOW_READ_STATUS_C", "INST_UPD_DTTM", "BUSINESS_DAYS_TO_HANDLE",
    "MessageText_clean",
]
missing_required = [c for c in required_cols if c not in messages_raw.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

print("Loaded message rows:", f"{len(messages_raw):,}")
print("Loaded source files:", len(source_file_summary))
print(source_file_summary.to_string(index=False))

## 4. Basic Message Cleaning

In [ ]:
messages = messages_raw.copy()

# Drop a legacy response-method column that was not carried into analysis.
messages = messages.drop(columns=["RESP_METH"], errors="ignore")

# Standardize key ID columns as strings. Empty strings are treated as missing.
for col in ["MESSAGE_ID", "DMRN", "FROM_USER_ID", "TO_USER_ID", "TOFROM_PAT"]:
    messages[col] = messages[col].astype("string").str.strip()
    messages.loc[messages[col].eq(""), col] = pd.NA

# In the source files, patient-side user IDs are sometimes blank even when TOFROM_PAT identifies the patient side.
mask_from_patient_inferred = messages["TOFROM_PAT"].eq("From Patient") & messages["FROM_USER_ID"].isna()
mask_to_patient_inferred = messages["TOFROM_PAT"].eq("To Patient") & messages["TO_USER_ID"].isna()
messages.loc[mask_from_patient_inferred, "FROM_USER_ID"] = "Patient"
messages.loc[mask_to_patient_inferred, "TO_USER_ID"] = "Patient"

# Parse time fields. These are central to response timing and later thread construction.
for col in ["CREATED_TIME", "FINAL_HANDLED_TIME", "INST_UPD_DTTM"]:
    messages[col] = pd.to_datetime(messages[col], errors="coerce")

messages["BUSINESS_DAYS_TO_HANDLE"] = pd.to_numeric(messages["BUSINESS_DAYS_TO_HANDLE"], errors="coerce")

for flag in ["NOTALLOW_REPLY_YN", "REPLY_DIRECT_YN"]:
    messages[flag] = normalize_yes_no(messages[flag]).astype("category")

messages["MESG_TYPE"] = messages["MESG_TYPE"].astype("string").str.strip()
messages["Prov_DEPT_For_Appt"] = messages["Prov_DEPT_For_Appt"].astype("string").str.strip()
messages["SUBJECT"] = messages["SUBJECT"].replace([None, ""], "Unknown").fillna("Unknown").astype("string")

messages["response_time"] = messages["FINAL_HANDLED_TIME"] - messages["CREATED_TIME"]
negative_response_time_rows = int((messages["response_time"].dt.total_seconds() < 0).sum())
messages.loc[messages["response_time"].dt.total_seconds() < 0, "response_time"] = pd.NaT
messages["update_delay"] = messages["INST_UPD_DTTM"] - messages["CREATED_TIME"]

messages["response_time_hours"] = messages["response_time"].dt.total_seconds() / 3600
messages["update_delay_hours"] = messages["update_delay"].dt.total_seconds() / 3600

timing_bins = [0, 1, 6, 24, 72, 168, float("inf")]
timing_labels = ["<1h", "1-6h", "6-24h", "1-3d", "3-7d", ">7d"]
messages["response_time_bin"] = pd.cut(messages["response_time_hours"], bins=timing_bins, labels=timing_labels, include_lowest=True)
messages["update_delay_bin"] = pd.cut(messages["update_delay_hours"], bins=timing_bins, labels=timing_labels, include_lowest=True)

basic_cleaning_qa = {
    "rows": int(len(messages)),
    "unique_message_ids": int(messages["MESSAGE_ID"].nunique(dropna=True)),
    "duplicate_message_id_rows": int(messages.duplicated("MESSAGE_ID", keep=False).sum()),
    "missing_dmrn": int(messages["DMRN"].isna().sum()),
    "missing_created_time": int(messages["CREATED_TIME"].isna().sum()),
    "inferred_from_patient_user_ids": int(mask_from_patient_inferred.sum()),
    "inferred_to_patient_user_ids": int(mask_to_patient_inferred.sum()),
    "negative_response_time_rows_set_missing": negative_response_time_rows,
    "negative_update_delay_rows": int((messages["update_delay_hours"] < 0).sum()),
}

print(json.dumps(basic_cleaning_qa, indent=2, default=str))

## 5. Production Subject Cleaning

In [ ]:
messages["SUBJECT_BASIC"] = messages["SUBJECT"].map(clean_subject_basic)
messages["SUBJECT_CLEAN"] = messages["SUBJECT_BASIC"].map(clean_topic_text)

# Symbol-only subjects do not carry semantic content. Use the message type to assign a stable topic label.
symbol_subjects = {
    "?", "??", "???", "????", "?????", "!", "!!", "!!!", "***",
    "%", ",", ":", ":(", ":)", ": )", "]", ":((", ":))",
}
symbol_mask = messages["SUBJECT_BASIC"].isin(symbol_subjects)
messages.loc[symbol_mask & messages["MESG_TYPE"].eq("Patient Medical Advice Request"), "SUBJECT_CLEAN"] = "medical advice request"
messages.loc[symbol_mask & messages["MESG_TYPE"].eq("User Message"), "SUBJECT_CLEAN"] = "user message"
messages.loc[symbol_mask & messages["MESG_TYPE"].eq("Case Reminder Message"), "SUBJECT_CLEAN"] = "case reminder"
messages.loc[symbol_mask & ~messages["MESG_TYPE"].isin([
    "Patient Medical Advice Request", "User Message", "Case Reminder Message"
]), "SUBJECT_CLEAN"] = "unknown"

messages["SUBJECT_CLEAN"] = messages["SUBJECT_CLEAN"].map(normalize_typos).map(normalize_known_subject_phrases)

subject_qa = {
    "unique_subject_raw": int(messages["SUBJECT"].nunique(dropna=True)),
    "unique_subject_basic": int(messages["SUBJECT_BASIC"].nunique(dropna=True)),
    "unique_subject_clean": int(messages["SUBJECT_CLEAN"].nunique(dropna=True)),
    "missing_subject_clean": int(messages["SUBJECT_CLEAN"].isna().sum()),
    "unknown_subject_clean": int(messages["SUBJECT_CLEAN"].eq("unknown").sum()),
    "symbol_subject_rows": int(symbol_mask.sum()),
    "rows_with_reply_forward_prefix_removed": int(messages["SUBJECT"].astype(str).str.contains(r"^(?:(?:re|fw|fwd)\s*:\s*)+", case=False, regex=True, na=False).sum()),
    "rows_with_encoding_artifact_after_cleaning": int(messages["SUBJECT_CLEAN"].astype("string").str.contains("Ã¢", na=False).sum()),
}

top_subject_clean = qa_counts(messages, "SUBJECT_CLEAN").head(25)

print(json.dumps(subject_qa, indent=2, default=str))
print(top_subject_clean.to_string(index=False))

## 6. Save Core Preprocessing Outputs

In [ ]:
messages_core_path = RUN_DIR / "messages_core_preprocessed.parquet"
source_summary_path = RUN_DIR / "message_source_file_summary.csv"
message_type_summary_path = RUN_DIR / "message_type_summary.csv"
subject_summary_path = RUN_DIR / "subject_clean_summary_top100.csv"
qa_summary_path = RUN_DIR / "qa_core_message_preprocessing_summary.json"

message_type_summary = qa_counts(messages, "MESG_TYPE")
subject_summary_top100 = qa_counts(messages, "SUBJECT_CLEAN").head(100)

messages.to_parquet(messages_core_path, index=False, compression="zstd")
source_file_summary.to_csv(source_summary_path, index=False)
message_type_summary.to_csv(message_type_summary_path, index=False)
subject_summary_top100.to_csv(subject_summary_path, index=False)

qa_summary = {
    "run_id": RUN_ID,
    "source_cleaned_dir": str(SOURCE_CLEANED_DIR),
    "output_dir": str(RUN_DIR),
    "stage": "core_message_preprocessing_before_provider_thread_demographics_cci_visits",
    "source_files": source_file_summary.to_dict(orient="records"),
    "basic_cleaning": basic_cleaning_qa,
    "subject_cleaning": subject_qa,
    "message_type_counts": message_type_summary.to_dict(orient="records"),
    "output_files": {
        "messages_core_preprocessed": str(messages_core_path),
        "message_source_file_summary": str(source_summary_path),
        "message_type_summary": str(message_type_summary_path),
        "subject_clean_summary_top100": str(subject_summary_path),
        "qa_summary": str(qa_summary_path),
    },
}

with open(qa_summary_path, "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2, default=str)

print("Saved:")
for path in [messages_core_path, source_summary_path, message_type_summary_path, subject_summary_path, qa_summary_path]:
    print(" -", path)

## 7. Build Clean Provider Master

In [ ]:
# Note: The provider master and supplemental user-mapping files are curated source inputs created before this preprocessing notebook.
# This notebook does not recreate the manual curation steps used to produce those files.
# It reads these files as fixed inputs and documents how they are merged/combined for sender/receiver classification.

provider_master_source_path = PROVIDER_SOURCE_DIR / "edited_provider_master.xlsx"
unmapped_users_path = PROVIDER_SOURCE_DIR / "unmapped_user_ids_updated.csv"
missing_users_path = PROVIDER_SOURCE_DIR / "MissingUsers.txt"

provider_master_source = pd.read_excel(provider_master_source_path, dtype=str)
provider_master_source = clean_columns(provider_master_source)

cols_to_drop = ["SPECIALTY_1", "SPECIALTY_2", "SPECIALTY_3", "ALL_SPECIALTIES", "SOURCE_USERS", "SOURCE_SENDERS"]
provider_master_base = provider_master_source.drop(columns=cols_to_drop, errors="ignore").copy()

for col in ["USER_ID", "PROV_NAME", "PROV_TYPE", "SPECIALTY", "PRIMARY_CARE_BUCKET", "CARE_BUCKET", "subspecialty", "NPI"]:
    if col not in provider_master_base.columns:
        provider_master_base[col] = pd.NA
    provider_master_base[col] = provider_master_base[col].astype("string").str.strip()
    provider_master_base.loc[provider_master_base[col].eq(""), col] = pd.NA

provider_master_base = provider_master_base.dropna(subset=["USER_ID"]).drop_duplicates("USER_ID", keep="first").copy()

unmapped = pd.read_csv(unmapped_users_path, dtype=str, encoding="cp1252", encoding_errors="replace")
unmapped = clean_columns(unmapped)

missing = pd.read_csv(
    missing_users_path,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
    sep=None,
    engine="python",
    on_bad_lines="skip",
)
missing = clean_columns(missing).rename(columns={
    "USER_ID": "PROV_ID",
    "USER_NAMES": "PROV_NAME",
    "USERSTATUS": "ACTIVE_STATUS",
})

common_cols = ["PROV_ID", "PROV_NAME", "PROV_TYPE", "IS_RESIDENT", "ACTIVE_STATUS", "LINE", "SPECIALTY", "NPI"]
for df_name, df in [("unmapped", unmapped), ("missing", missing)]:
    for col in common_cols:
        if col not in df.columns:
            df[col] = pd.NA

unmapped = unmapped[common_cols].copy()
missing = missing[common_cols].copy()
for df in [unmapped, missing]:
    for col in common_cols:
        df[col] = df[col].astype("string").str.strip()
        df.loc[df[col].eq(""), col] = pd.NA

combined_user_supplement = (
    unmapped.set_index("PROV_ID")
    .combine_first(missing.set_index("PROV_ID"))
    .reset_index()
    .dropna(subset=["PROV_ID"])
    .drop_duplicates("PROV_ID", keep="first")
)

# Fill missing provider names/types/specialties/NPI from the supplement when possible, then append truly new IDs.
provider_master_clean = provider_master_base.copy()
supplement_lookup = combined_user_supplement.drop_duplicates("PROV_ID").set_index("PROV_ID")
provider_master_clean = provider_master_clean.set_index("USER_ID")

fill_map = {
    "PROV_NAME": "PROV_NAME",
    "PROV_TYPE": "PROV_TYPE",
    "SPECIALTY": "SPECIALTY",
    "NPI": "NPI",
}
provider_existing_ids = set(provider_master_clean.index.astype(str))
supplement_ids = set(supplement_lookup.index.astype(str))

filled_counts = {}
for target_col, source_col in fill_map.items():
    before_missing = provider_master_clean[target_col].isna() | provider_master_clean[target_col].eq("")
    provider_master_clean[target_col] = provider_master_clean[target_col].fillna(supplement_lookup[source_col])
    after_missing = provider_master_clean[target_col].isna() | provider_master_clean[target_col].eq("")
    filled_counts[target_col] = int(before_missing.sum() - after_missing.sum())

provider_master_clean = provider_master_clean.reset_index()

new_supplement_rows = combined_user_supplement.loc[
    ~combined_user_supplement["PROV_ID"].astype(str).isin(provider_existing_ids)
].copy()
new_provider_rows = pd.DataFrame({
    "USER_ID": new_supplement_rows["PROV_ID"],
    "PROV_NAME": new_supplement_rows["PROV_NAME"],
    "PROV_TYPE": new_supplement_rows["PROV_TYPE"],
    "SPECIALTY": new_supplement_rows["SPECIALTY"],
    "PRIMARY_CARE_BUCKET": pd.NA,
    "CARE_BUCKET": pd.NA,
    "subspecialty": pd.NA,
    "NPI": new_supplement_rows["NPI"],
})

provider_master_clean = pd.concat([provider_master_clean, new_provider_rows], ignore_index=True)
provider_master_clean = provider_master_clean.drop_duplicates("USER_ID", keep="first").reset_index(drop=True)

# Restore the provider role buckets from the original preprocessing notebook.
# These categories are the structured starting point for SENDER_CATEGORY and RECEIVER_CATEGORY.
role_mapping = {
    "Patient": "Patient",
    "Physician": "Physician",
    "Resident": "Physician",
    "Fellow": "Physician",
    "Anesthesiologist": "Physician",
    "DO": "Physician",
    "Registered Nurse": "Nurse",
    "Case Manager RN": "Nurse",
    "Care Manager RN": "Nurse",
    "Clinical Care Manager RN": "Nurse",
    "Nurse Practitioner": "APP",
    "Nurse Midwife": "APP",
    "Midwife": "APP",
    "Certified Nurse Midwife": "APP",
    "CNM": "APP",
    "Nurse Anesthetist": "APP",
    "Physician Assistant": "APP",
    "Advanced Practice Provider": "APP",
    "Clinical Nurse Specialist": "APP",
    "Medical Assistant": "Support Staff",
    "Patient Care Associate": "Support Staff",
    "Patient Service Specialist": "Support Staff",
    "Support Staff": "Support Staff",
    "Scribe": "Support Staff",
    "Unit Clerk": "Support Staff",
    "Licensed Nurse": "Support Staff",
    "Medical Secretary": "Support Staff",
    "Patient Access Coordinator": "Support Staff",
    "Supervisor Patient Access": "Support Staff",
    "Psychiatrist": "Behavioral Health",
    "Psychology": "Behavioral Health",
    "Psychologist": "Behavioral Health",
    "Addiction Counselor": "Behavioral Health",
    "Addiction Psychiatry": "Behavioral Health",
    "Social Worker": "Behavioral Health",
    "Counselor": "Behavioral Health",
    "Psychology Fellow": "Behavioral Health",
    "LPCC-S": "Behavioral Health",
    "Physical Therapist": "Therapist",
    "Occupational Therapist": "Therapist",
    "Speech and Language Pathologist": "Therapist",
    "Athletic Trainer": "Therapist",
    "Massage Therapist": "Therapist",
    "Music Therapist": "Therapist",
    "Occupational Therapy Assistant": "Therapist",
    "Recreational Therapist": "Therapist",
    "Therapist": "Therapist",
    "Respiratory Therapist": "Therapist",
    "Art Therapist": "Therapist",
    "Genetic Counselor": "Allied Health",
    "Physical Therapy Assistant": "Therapist",
    "Acupuncturist": "Therapist",
    "M.A., CF-SLP": "Therapist",
    "Pharmacist": "Allied Health",
    "Pharmacy Tech": "Allied Health",
    "Audiologist": "Allied Health",
    "Registered Dietitian": "Allied Health",
    "Optometrist": "Allied Health",
    "Chiropractor": "Allied Health",
    "Dentist": "Allied Health",
    "Student Dietitian": "Allied Health",
    "Dietitian": "Allied Health",
    "Chaplain": "Allied Health",
    "Exercise Physiologist": "Allied Health",
    "Rehabilitation Engineer": "Allied Health",
    "Lab Technician": "Tech Specialist",
    "Radiology Technician": "Tech Specialist",
    "Polysomnographic Tech": "Tech Specialist",
    "Registered Polysomnographic Tech": "Tech Specialist",
    "Emergency Medical Technician": "Tech Specialist",
    "Dietary Technician": "Tech Specialist",
    "Rad Technologist": "Tech Specialist",
    "Technologist": "Tech Specialist",
    "Exercise Physiology Tech": "Tech Specialist",
    "Perfusionist": "Tech Specialist",
    "Sonographer": "Tech Specialist",
    "Cath Lab Tech": "Tech Specialist",
    "Research Coordinator": "Administration",
    "Case Manager": "Administration",
    "Coder/Biller": "Administration",
    "Care Coordinator / Patient Navigator": "Administration",
    "Care Coordinator": "Administration",
    "Care Manager": "Administration",
    "Patient Navigator": "Administration",
    "Genetic Counselor Student": "Student",
    "Medical Assistant Student": "Student",
    "Medical Student": "Student",
    "Mid Level Student": "Student",
    "Nursing Student": "Student",
    "Occupational Therapy Student": "Student",
    "Pharmacy Student": "Student",
    "Physical Therapy Student": "Student",
    "Psychology Student": "Student",
    "Social Work Student": "Student",
    "Speech Therapy Student": "Student",
    "Unknown": "Unknown",
    "Unknow": "Unknown",
}

provider_master_clean["Type_BUCKET"] = (
    provider_master_clean["PROV_TYPE"]
    .map(role_mapping)
    .fillna("Unknown")
)

provider_master_qa = {
    "provider_master_source_rows": int(len(provider_master_source)),
    "provider_master_base_rows": int(len(provider_master_base)),
    "combined_user_supplement_rows": int(len(combined_user_supplement)),
    "supplement_ids_already_in_provider_master": int(len(provider_existing_ids & supplement_ids)),
    "new_provider_rows_added_from_supplement": int(len(new_provider_rows)),
    "provider_master_clean_rows": int(len(provider_master_clean)),
    "provider_master_clean_unique_user_ids": int(provider_master_clean["USER_ID"].nunique(dropna=True)),
    "missing_provider_name": int(provider_master_clean["PROV_NAME"].isna().sum()),
    "missing_provider_type": int(provider_master_clean["PROV_TYPE"].isna().sum()),
    "missing_specialty": int(provider_master_clean["SPECIALTY"].isna().sum()),
    "filled_counts_from_supplement": filled_counts,
    "type_bucket_counts": provider_master_clean["Type_BUCKET"].value_counts(dropna=False).to_dict(),
}

provider_type_bucket_counts = qa_counts(provider_master_clean, "Type_BUCKET")

print(json.dumps(provider_master_qa, indent=2, default=str))
print(provider_type_bucket_counts.to_string(index=False))

## 8. Enrich Sender And Receiver Provider Fields

In [ ]:
# Merge provider metadata onto sender and receiver sides. Patient-side IDs remain explicitly labeled as Patient.
provider_for_merge = provider_master_clean[[
    "USER_ID", "PROV_NAME", "PROV_TYPE", "SPECIALTY", "PRIMARY_CARE_BUCKET", "CARE_BUCKET", "subspecialty", "NPI", "Type_BUCKET"
]].drop_duplicates("USER_ID")

def enrich_provider_side(df: pd.DataFrame, side: str, user_col: str) -> pd.DataFrame:
    provider_prefixed = provider_for_merge.add_prefix(f"{side}_")
    out = df.merge(
        provider_prefixed,
        left_on=user_col,
        right_on=f"{side}_USER_ID",
        how="left",
    )
    return out.drop(columns=[f"{side}_USER_ID"], errors="ignore")

messages_provider_enriched = messages.copy()
messages_provider_enriched = enrich_provider_side(messages_provider_enriched, "from", "FROM_USER_ID")
messages_provider_enriched = enrich_provider_side(messages_provider_enriched, "to", "TO_USER_ID")

messages_provider_enriched["PROV_NAME"] = (
    messages_provider_enriched[["from_PROV_NAME", "to_PROV_NAME"]]
    .fillna("")
    .agg(lambda row: ", ".join([value for value in row if value]), axis=1)
    .replace("", pd.NA)
)

messages_provider_enriched = messages_provider_enriched.rename(columns={
    "from_PROV_TYPE": "SENDER_TYPE",
    "to_PROV_TYPE": "RECEIVER_TYPE",
    "from_Type_BUCKET": "SENDER_CATEGORY",
    "to_Type_BUCKET": "RECEIVER_CATEGORY",
    "from_SPECIALTY": "from_SPECIALTY",
    "to_SPECIALTY": "to_SPECIALTY",
    "from_PRIMARY_CARE_BUCKET": "from_PRIMARY_CARE_BUCKET",
    "to_PRIMARY_CARE_BUCKET": "to_PRIMARY_CARE_BUCKET",
    "from_CARE_BUCKET": "from_CARE_BUCKET",
    "to_CARE_BUCKET": "to_CARE_BUCKET",
    "from_subspecialty": "from_subspecialty",
    "to_subspecialty": "to_subspecialty",
})

messages_provider_enriched = messages_provider_enriched.drop(columns=["from_PROV_NAME", "to_PROV_NAME"], errors="ignore")

messages_provider_enriched.loc[
    messages_provider_enriched["SENDER_TYPE"].isna() & messages_provider_enriched["FROM_USER_ID"].eq("Patient"),
    "SENDER_TYPE",
] = "Patient"
messages_provider_enriched.loc[
    messages_provider_enriched["RECEIVER_TYPE"].isna() & messages_provider_enriched["TO_USER_ID"].eq("Patient"),
    "RECEIVER_TYPE",
] = "Patient"
messages_provider_enriched.loc[
    messages_provider_enriched["SENDER_CATEGORY"].isna() & messages_provider_enriched["FROM_USER_ID"].eq("Patient"),
    "SENDER_CATEGORY",
] = "Patient"
messages_provider_enriched.loc[
    messages_provider_enriched["RECEIVER_CATEGORY"].isna() & messages_provider_enriched["TO_USER_ID"].eq("Patient"),
    "RECEIVER_CATEGORY",
] = "Patient"

# Restore the original structured system relabeling that was applied before
# the detailed sender/receiver refinement rules.
system_ids = ["MYCHARTG", "ADTBOTSERVICE2", "RPABOTSERVICE", "AFTERVISIT"]
messages_provider_enriched.loc[
    messages_provider_enriched["FROM_USER_ID"].isin(system_ids)
    | messages_provider_enriched["MESG_TYPE"].eq("System Message"),
    ["SENDER_TYPE", "SENDER_CATEGORY"],
] = "System"
messages_provider_enriched.loc[
    messages_provider_enriched["TO_USER_ID"].isin(system_ids),
    ["RECEIVER_TYPE", "RECEIVER_CATEGORY"],
] = "System"

sender_nonpatient_id = messages_provider_enriched["FROM_USER_ID"].notna() & ~messages_provider_enriched["FROM_USER_ID"].eq("Patient")
receiver_nonpatient_id = messages_provider_enriched["TO_USER_ID"].notna() & ~messages_provider_enriched["TO_USER_ID"].eq("Patient")

provider_enrichment_qa = {
    "message_rows": int(len(messages_provider_enriched)),
    "sender_nonpatient_id_rows": int(sender_nonpatient_id.sum()),
    "receiver_nonpatient_id_rows": int(receiver_nonpatient_id.sum()),
    "sender_nonpatient_matched_type": int((sender_nonpatient_id & messages_provider_enriched["SENDER_TYPE"].notna()).sum()),
    "receiver_nonpatient_matched_type": int((receiver_nonpatient_id & messages_provider_enriched["RECEIVER_TYPE"].notna()).sum()),
    "sender_nonpatient_missing_type": int((sender_nonpatient_id & messages_provider_enriched["SENDER_TYPE"].isna()).sum()),
    "receiver_nonpatient_missing_type": int((receiver_nonpatient_id & messages_provider_enriched["RECEIVER_TYPE"].isna()).sum()),
    "sender_patient_rows": int(messages_provider_enriched["FROM_USER_ID"].eq("Patient").sum()),
    "receiver_patient_rows": int(messages_provider_enriched["TO_USER_ID"].eq("Patient").sum()),
    "rows_with_any_provider_name": int(messages_provider_enriched["PROV_NAME"].notna().sum()),
}

sender_category_counts = qa_counts(messages_provider_enriched, "SENDER_CATEGORY")
receiver_category_counts = qa_counts(messages_provider_enriched, "RECEIVER_CATEGORY")

print(json.dumps(provider_enrichment_qa, indent=2, default=str))
print("Sender categories:")
print(sender_category_counts.to_string(index=False))
print("Receiver categories:")
print(receiver_category_counts.to_string(index=False))

## 9. Save Provider Enrichment Outputs

In [ ]:
provider_master_path = RUN_DIR / "provider_master_clean.parquet"
messages_provider_enriched_path = RUN_DIR / "messages_provider_enriched.parquet"
provider_type_counts_path = RUN_DIR / "provider_type_bucket_counts.csv"
sender_category_counts_path = RUN_DIR / "sender_category_counts_initial.csv"
receiver_category_counts_path = RUN_DIR / "receiver_category_counts_initial.csv"
provider_qa_summary_path = RUN_DIR / "qa_provider_enrichment_summary.json"

provider_master_clean.to_parquet(provider_master_path, index=False)
messages_provider_enriched.to_parquet(messages_provider_enriched_path, index=False, compression="zstd")
provider_type_bucket_counts.to_csv(provider_type_counts_path, index=False)
sender_category_counts.to_csv(sender_category_counts_path, index=False)
receiver_category_counts.to_csv(receiver_category_counts_path, index=False)

provider_qa_summary = {
    "run_id": RUN_ID,
    "provider_source_dir": str(PROVIDER_SOURCE_DIR),
    "provider_master_source_path": str(provider_master_source_path),
    "unmapped_users_path": str(unmapped_users_path),
    "missing_users_path": str(missing_users_path),
    "provider_master": provider_master_qa,
    "provider_enrichment": provider_enrichment_qa,
    "output_files": {
        "provider_master_clean": str(provider_master_path),
        "messages_provider_enriched": str(messages_provider_enriched_path),
        "provider_type_bucket_counts": str(provider_type_counts_path),
        "sender_category_counts_initial": str(sender_category_counts_path),
        "receiver_category_counts_initial": str(receiver_category_counts_path),
        "provider_qa_summary": str(provider_qa_summary_path),
    },
}

with open(provider_qa_summary_path, "w", encoding="utf-8") as f:
    json.dump(provider_qa_summary, f, indent=2, default=str)

print("Saved provider enrichment outputs:")
for path in [
    provider_master_path,
    messages_provider_enriched_path,
    provider_type_counts_path,
    sender_category_counts_path,
    receiver_category_counts_path,
    provider_qa_summary_path,
]:
    print(" -", path)

## 10. Final Analytic Sender, Receiver, and Origin

This section creates only the sender/source variables used downstream in the final analysis:

- `Sender_all`
- `Receiver_all`
- `origin`

The logic starts from structured sender/receiver categories. It then applies the limited system-identification rules that can change the final analytic origin from Provider to System. Broader workflow labels used during development are intentionally omitted from the main preprocessing pipeline because they are not analytic variables in the final analysis.


### 10.1 Start From Structured Categories

`Sender_all` and `Receiver_all` begin as copies of the structured sender and receiver categories. System-specific user IDs and `MESG_TYPE == "System Message"` are treated as structured system identifiers.


In [ ]:
system_ids = ['MYCHARTG', 'ADTBOTSERVICE2', 'RPABOTSERVICE', 'AFTERVISIT']

messages_refined = messages_provider_enriched.copy()

messages_refined['Sender_all'] = messages_refined['SENDER_CATEGORY']
messages_refined['Receiver_all'] = messages_refined['RECEIVER_CATEGORY']

structured_system_sender = (
    messages_refined['FROM_USER_ID'].isin(system_ids)
    | messages_refined['MESG_TYPE'].eq('System Message')
)
structured_system_receiver = messages_refined['TO_USER_ID'].isin(system_ids)

messages_refined.loc[structured_system_sender, 'Sender_all'] = 'System'
messages_refined.loc[structured_system_receiver, 'Receiver_all'] = 'System'

messages_refined['origin_original'] = np.select(
    [
        messages_refined['Sender_all'].astype(str).eq('System'),
        messages_refined['Sender_all'].astype(str).eq('Patient'),
    ],
    ['System', 'Patient'],
    default='Provider'
)


### 10.2 Limited System-Origin Rules

The following rules identify standardized automated communications that were not fully captured by structured metadata. These rules are used only to assign final analytic source/origin; they do not analyze message body text for clinical content, symptoms, topics, sentiment, or reply intent.


In [ ]:
known_system_subjects = [
    'covid 19 update from your pcp please do not reply to message',
    'covid 19 update from your pcp please do not reply to this message',
    'saturday flu vaccination appointments now available',
    'flu vaccine reminder',
    'covid-19/omicron update',
    'osu stoneridge gim office relocating to 6700 university boulevard in dublin'
    'insurance issue',
]

known_system_subject_sender = (
    messages_refined['SUBJECT_CLEAN'].astype(str).str.lower().isin(known_system_subjects)
    & messages_refined['RECEIVER_TYPE'].astype(str).str.lower().eq('patient')
)

system_broadcast_sender = (
    (
        messages_refined['SUBJECT_CLEAN'].str.contains(
            r"\bcovid(?: |-)?(?:19)?\b.{0,30}\b(?:update|reminder|information|alert|message)\b",
            case=False, na=False
        )
        | messages_refined['SUBJECT_CLEAN'].str.contains(r"relocat(?:ed|ion|ing|e)?", case=False, na=False)
        | messages_refined['SUBJECT_CLEAN'].str.contains(
            r"(?:office|clinic|location|practice|site).{0,30}\bclos(?:e|ing|ure)\b",
            case=False, na=False
        )
        | messages_refined['SUBJECT_CLEAN'].str.contains(r"stoneridge closing", case=False, na=False)
        | messages_refined['SUBJECT_CLEAN'].str.contains(r"(?:office|clinic|location).*\bmoved?\b", case=False, na=False)
        | messages_refined['SUBJECT_CLEAN'].str.contains(
            r"(?:opening|grand opening|important notice|service interruption|downtime notice|health care service interruption)",
            case=False, na=False
        )
        | messages_refined['MessageText_clean'].str.contains(
            r"(?:service disruption|scheduled maintenance|we are pleased to announce the opening|health care service interruption)",
            case=False, na=False
        )
        | messages_refined['MessageText_clean'].str.contains('please do not respond to this message', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('do not reply', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('this is an automated message', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('this is a system generated message', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('automated notification', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('system generated notification', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('YOU HAVE BEEN SCHEDULED FOR', case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains('reminder for your upcoming procedure', case=False, na=False)
    )
    & messages_refined['RECEIVER_TYPE'].astype(str).str.lower().eq('patient')
    & ~messages_refined['SENDER_TYPE'].astype(str).str.lower().eq('system')
    & messages_refined['SENDER_TYPE'].notna()
    & ~messages_refined['SENDER_TYPE'].eq('')
)

system_cancellation_confirmation_sender = (
    messages_refined['SUBJECT_CLEAN'].str.contains(r"appointment cancellation request", case=False, na=False)
    & messages_refined['MessageText_clean'].str.contains(
        r"we have received your request to cancel.*(?:appointment|radiology appointment).*?(?:completed|processed|confirmed)",
        case=False, na=False
    )
    & (messages_refined['SENDER_TYPE'].isna() | messages_refined['SENDER_TYPE'].eq(''))
)

system_procedure_reminder_sender = (
    messages_refined['MESG_TYPE'].astype(str).str.lower().eq('case reminder message')
    & messages_refined['Receiver_all'].astype(str).str.lower().eq('patient')
    & messages_refined['MessageText_clean'].str.contains(
        r"(?:reminder for your upcoming procedure|prior to your upcoming procedure, please view the following educational video|You are scheduled to receive a hip replacement|You are scheduled to receive a knee replacement)",
        case=False, na=False
    )
)

automated_system_sender = (
    known_system_subject_sender
    | system_broadcast_sender
    | system_cancellation_confirmation_sender
    | system_procedure_reminder_sender
)
automated_system_sender = automated_system_sender & ~messages_refined['origin_original'].eq('Patient')

messages_refined.loc[automated_system_sender, 'Sender_all'] = 'System'


### 10.3 Receiver Rules Needed for Final Outputs

`Receiver_all` is used in the thread summary outputs. The notebook keeps only receiver rules that identify system receivers or direct automated confirmations needed for the final analytic sender/receiver fields.


In [ ]:
appointment_cancel_missing_receiver = (
    messages_refined['SUBJECT_CLEAN'].str.contains(r"appointment cancellation request", case=False, na=False)
    & messages_refined['SENDER_TYPE'].astype(str).str.lower().eq('patient')
    & (messages_refined['TO_USER_ID'].isna() | messages_refined['TO_USER_ID'].eq(''))
)

appointment_request_missing_receiver = (
    messages_refined['SUBJECT_CLEAN'].str.contains(r"appointment request", case=False, na=False)
    & messages_refined['SENDER_TYPE'].astype(str).str.lower().eq('patient')
    & (messages_refined['TO_USER_ID'].isna() | messages_refined['TO_USER_ID'].eq(''))
)

med_refill_system_receiver = (
    messages_refined['MESG_TYPE'].astype(str).str.lower().eq('patient medication renewal request')
    & messages_refined['SENDER_TYPE'].astype(str).str.lower().eq('patient')
    & (
        messages_refined['TO_USER_ID'].isna()
        | messages_refined['TO_USER_ID'].eq('')
        | messages_refined['TO_USER_ID'].eq('System')
    )
)

patient_misc_refill_system_receiver = (
    messages_refined['MESG_TYPE'].astype(str).str.lower().eq('patient medication renewal request')
    & messages_refined['SENDER_TYPE'].astype(str).str.lower().eq('patient')
    & (messages_refined['TO_USER_ID'].isna() | messages_refined['TO_USER_ID'].eq(''))
    & (
        messages_refined['SUBJECT_CLEAN'].str.contains(r"(?:medication|refill|prescription|rx|renew)", case=False, na=False)
        | messages_refined['MessageText_clean'].str.contains(r"\b(?:refill|prescription|medication|rx|renew)\b", case=False, na=False)
    )
)

system_receiver = (
    appointment_cancel_missing_receiver
    | appointment_request_missing_receiver
    | med_refill_system_receiver
    | patient_misc_refill_system_receiver
)

messages_refined.loc[system_receiver, 'Receiver_all'] = 'System'
messages_refined.loc[system_cancellation_confirmation_sender, 'Receiver_all'] = 'Patient'


### 10.4 Final Origin and Aggregate QA

Final analytic origin is defined from `Sender_all`: System if the sender is System, Patient if the sender is Patient, and Provider otherwise.


In [ ]:
messages_refined['Sender_all'] = messages_refined['Sender_all'].astype(str)
messages_refined['Receiver_all'] = messages_refined['Receiver_all'].astype(str)

messages_refined.loc[
    messages_refined['Sender_all'].isna()
    | messages_refined['Sender_all'].eq('nan')
    | messages_refined['Sender_all'].eq(''),
    'Sender_all'
] = 'Unknown'

messages_refined.loc[
    messages_refined['Receiver_all'].isna()
    | messages_refined['Receiver_all'].eq('nan')
    | messages_refined['Receiver_all'].eq(''),
    'Receiver_all'
] = 'Unknown'

messages_refined['origin'] = np.select(
    [
        messages_refined['Sender_all'].eq('System'),
        messages_refined['Sender_all'].eq('Patient'),
    ],
    ['System', 'Patient'],
    default='Provider'
)

sender_all_counts = qa_counts(messages_refined, 'Sender_all')
receiver_all_counts = qa_counts(messages_refined, 'Receiver_all')
origin_original_counts = qa_counts(messages_refined, 'origin_original')
origin_counts = qa_counts(messages_refined, 'origin')

origin_transition_matrix = pd.crosstab(
    messages_refined['origin_original'],
    messages_refined['origin'],
    margins=True,
)

origin_changed = messages_refined['origin_original'].ne(messages_refined['origin'])
patient_origin_changed = messages_refined['origin_original'].eq('Patient').ne(messages_refined['origin'].eq('Patient'))
provider_to_system_change = messages_refined['origin_original'].eq('Provider') & messages_refined['origin'].eq('System')
all_changes_provider_to_system = bool((~origin_changed | provider_to_system_change).all())

system_rule_summary = pd.DataFrame([
    {'rule': 'structured_system_sender', 'rows': int(structured_system_sender.sum())},
    {'rule': 'known_system_subject_sender', 'rows': int(known_system_subject_sender.sum())},
    {'rule': 'system_broadcast_sender', 'rows': int(system_broadcast_sender.sum())},
    {'rule': 'system_cancellation_confirmation_sender', 'rows': int(system_cancellation_confirmation_sender.sum())},
    {'rule': 'system_procedure_reminder_sender', 'rows': int(system_procedure_reminder_sender.sum())},
    {'rule': 'any_automated_system_sender', 'rows': int(automated_system_sender.sum())},
    {'rule': 'system_receiver', 'rows': int(system_receiver.sum())},
])

origin_audit_summary = {
    'message_rows': int(len(messages_refined)),
    'origin_changed_rows': int(origin_changed.sum()),
    'origin_changed_percent': round(float(origin_changed.mean() * 100), 3),
    'provider_to_system_rows': int(provider_to_system_change.sum()),
    'all_origin_changes_provider_to_system': all_changes_provider_to_system,
    'patient_origin_changed_rows': int(patient_origin_changed.sum()),
    'original_origin_counts': messages_refined['origin_original'].value_counts(dropna=False).to_dict(),
    'final_origin_counts': messages_refined['origin'].value_counts(dropna=False).to_dict(),
    'system_identification_rule_counts': system_rule_summary.to_dict(orient='records'),
}

origin_audit_table = pd.DataFrame([
    {
        'qa_item': 'Messages with changed analytic origin',
        'rows': int(origin_changed.sum()),
        'percent_of_messages': round(float(origin_changed.mean() * 100), 3),
        'interpretation': 'Rows whose final analytic origin differs from structured-origin classification.',
    },
    {
        'qa_item': 'Provider-to-System origin changes',
        'rows': int(provider_to_system_change.sum()),
        'percent_of_messages': round(float(provider_to_system_change.mean() * 100), 3),
        'interpretation': 'System communications identified by limited automated-message rules.',
    },
    {
        'qa_item': 'All origin changes were Provider-to-System',
        'rows': int(origin_changed.sum()) if all_changes_provider_to_system else 0,
        'percent_of_messages': round(float(origin_changed.mean() * 100), 3) if all_changes_provider_to_system else 0,
        'interpretation': str(all_changes_provider_to_system),
    },
    {
        'qa_item': 'Patient-origin changes',
        'rows': int(patient_origin_changed.sum()),
        'percent_of_messages': round(float(patient_origin_changed.mean() * 100), 3),
        'interpretation': 'Expected to be zero; Patient origin is preserved from structured metadata.',
    },
])

sender_receiver_classification_qa = {
    'message_rows': int(len(messages_refined)),
    'origin_changed_rows': int(origin_changed.sum()),
    'receiver_all_system_rows': int(messages_refined['Receiver_all'].eq('System').sum()),
    'origin_audit': origin_audit_summary,
    'sender_all_counts': messages_refined['Sender_all'].value_counts(dropna=False).to_dict(),
    'receiver_all_counts': messages_refined['Receiver_all'].value_counts(dropna=False).to_dict(),
}

print('Original structured origin counts:')
print(origin_original_counts.to_string(index=False))
print('Final analytic origin counts:')
print(origin_counts.to_string(index=False))
print('Origin transition matrix:')
print(origin_transition_matrix.to_string())
print('Origin audit table:')
print(origin_audit_table.to_string(index=False))


## 11. Save Final Pre-Thread Message Dataset

This is the handoff point for downstream notebooks. The output `messages_without_threads.parquet` contains the fully cleaned and sender/source-classified message-level data, but it does not contain `thread_id` or other thread-derived variables.

Downstream notebooks should use this file as input:

- `02_threading_diagnostics_sensitivity.ipynb` reads `messages_without_threads.parquet` and creates 15-day, 30-day, and 60-day threaded datasets.
- `04_build_demographics_cci_clean.ipynb` reads the same message IDs, patient IDs, and timestamps to compute message-time Charlson CCI.

The timestamped run-folder copy preserves reproducibility. The stable working copy gives the later notebooks a consistent file name.


In [ ]:
messages_without_threads = messages_refined.copy()

messages_without_threads_path = RUN_DIR / "messages_without_threads.parquet"
sender_all_counts_path = RUN_DIR / "sender_all_counts.csv"
receiver_all_counts_path = RUN_DIR / "receiver_all_counts.csv"
origin_original_counts_path = RUN_DIR / "origin_original_counts.csv"
origin_counts_path = RUN_DIR / "origin_counts.csv"
origin_transition_matrix_path = RUN_DIR / "origin_transition_matrix.csv"
origin_audit_table_path = RUN_DIR / "origin_audit_table.csv"
sender_receiver_qa_summary_path = RUN_DIR / "qa_sender_source_classification_summary.json"

messages_without_threads.to_parquet(messages_without_threads_path, index=False, compression="zstd")

canonical_messages_without_threads_path = None
if WRITE_CANONICAL_MESSAGES_WITHOUT_THREADS:
    WORKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    canonical_messages_without_threads_path = MESSAGES_WITHOUT_THREADS_CANONICAL_PATH
    messages_without_threads.to_parquet(canonical_messages_without_threads_path, index=False, compression="zstd")

sender_all_counts.to_csv(sender_all_counts_path, index=False)
receiver_all_counts.to_csv(receiver_all_counts_path, index=False)
origin_original_counts.to_csv(origin_original_counts_path, index=False)
origin_counts.to_csv(origin_counts_path, index=False)
origin_transition_matrix.to_csv(origin_transition_matrix_path)
origin_audit_table.to_csv(origin_audit_table_path, index=False)

sender_receiver_qa_summary = {
    "run_id": RUN_ID,
    "input_messages_provider_enriched": str(messages_provider_enriched_path),
    "classification_qa": sender_receiver_classification_qa,
    "handoff_note": "messages_without_threads.parquet is the final pre-thread message-level dataset for downstream threading and CCI notebooks.",
    "output_files": {
        "messages_without_threads_run_copy": str(messages_without_threads_path),
        "messages_without_threads_canonical": str(canonical_messages_without_threads_path) if canonical_messages_without_threads_path is not None else None,
        "sender_all_counts": str(sender_all_counts_path),
        "receiver_all_counts": str(receiver_all_counts_path),
        "origin_original_counts": str(origin_original_counts_path),
        "origin_counts": str(origin_counts_path),
        "origin_transition_matrix": str(origin_transition_matrix_path),
        "origin_audit_table": str(origin_audit_table_path),
        "sender_source_classification_qa_summary": str(sender_receiver_qa_summary_path),
    },
}

with open(sender_receiver_qa_summary_path, "w", encoding="utf-8") as f:
    json.dump(sender_receiver_qa_summary, f, indent=2, default=str)

print("Saved final pre-thread message dataset and sender/source QA outputs:")
for path in [
    messages_without_threads_path,
    canonical_messages_without_threads_path,
    sender_all_counts_path,
    receiver_all_counts_path,
    origin_original_counts_path,
    origin_counts_path,
    origin_transition_matrix_path,
    origin_audit_table_path,
    sender_receiver_qa_summary_path,
]:
    if path is not None:
        print(" -", path)


## 12. Downstream Threading Handoff

Thread construction is intentionally handled outside this preprocessing notebook.

The downstream threading notebook consumes `messages_without_threads.parquet` and creates the thread-derived datasets and diagnostics:

- 15-day, 30-day, and 60-day thread-gap versions
- intermessage gap summaries and ECDF diagnostics
- message-level threaded outputs
- thread-level summary outputs
- threading sensitivity comparison tables

This separation keeps preprocessing focused on message cleaning and sender/source classification, while making thread-definition diagnostics reproducible in one dedicated notebook.


## 13. Data Dictionary for Created Variables

This data dictionary documents variables created in this preprocessing notebook. Thread-derived variables such as `thread_id`, `starts_new_thread`, and `gap_days_same_key` are documented in the separate threading diagnostics notebook.


In [ ]:
final_variable_dictionary = pd.DataFrame(
    [
        {
            "variable": "SUBJECT_CLEAN",
            "level": "message",
            "definition": "Normalized message subject after prefix removal, lowercasing, typo cleanup, and standardized subject replacements.",
            "created_in_section": "Basic cleaning and subject normalization",
        },
        {
            "variable": "Sender_all",
            "level": "message",
            "definition": "Final analytic sender category. Starts from structured SENDER_CATEGORY and applies the original sender/source refinement logic used to create the analytic dataset.",
            "created_in_section": "Final Analytic Sender, Receiver, and Origin",
        },
        {
            "variable": "Receiver_all",
            "level": "message",
            "definition": "Final analytic receiver category. Starts from structured RECEIVER_CATEGORY and applies the original receiver/source refinement logic used to create the analytic dataset.",
            "created_in_section": "Final Analytic Sender, Receiver, and Origin",
        },
        {
            "variable": "origin",
            "level": "message",
            "definition": "Final analytic message-origin category: System when Sender_all is System, Patient when Sender_all is Patient, otherwise Provider.",
            "created_in_section": "Final Analytic Sender, Receiver, and Origin",
        },
        {
            "variable": "message_origin_category",
            "level": "message",
            "definition": "Message-origin category retained for compatibility with prior analytic code; aligned with the final analytic sender/source classification.",
            "created_in_section": "Final Analytic Sender, Receiver, and Origin",
        },
        {
            "variable": "origin_original",
            "level": "message",
            "definition": "Audit comparator origin calculated from structured sender/source information before final system-identification rules are applied.",
            "created_in_section": "Final Analytic Sender, Receiver, and Origin",
        },
        {
            "variable": "messages_without_threads",
            "level": "dataset",
            "definition": "Final pre-thread message-level dataset saved as messages_without_threads.parquet for downstream threading, demographics/CCI, visit/encounter, and analytic merge notebooks.",
            "created_in_section": "Save Final Pre-Thread Message Dataset",
        },
    ]
)

final_variable_dictionary_path = RUN_DIR / "final_variable_dictionary.csv"
final_variable_dictionary.to_csv(final_variable_dictionary_path, index=False)

print("Saved final variable dictionary:")
print(" -", final_variable_dictionary_path)
print(final_variable_dictionary[["variable", "level", "created_in_section"]].to_string(index=False))


## 14. Final Reproducibility Summary

This section writes a machine-readable summary of the runtime environment, configured input file names, output file names, and row counts at major stages.

Privacy note: row-level message outputs are restricted EHR-derived records and should not be publicly shared. Shareable materials should include code and aggregate QA summaries only.


In [ ]:
def package_version(package_name: str) -> str:
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return "not_installed"


key_output_files = {
    "messages_without_threads_run_copy": str(messages_without_threads_path),
    "messages_without_threads_canonical": str(canonical_messages_without_threads_path) if canonical_messages_without_threads_path is not None else None,
    "core_message_preprocessing_qa": str(qa_summary_path),
    "provider_enrichment_qa": str(provider_qa_summary_path),
    "sender_source_classification_qa": str(sender_receiver_qa_summary_path),
    "origin_transition_matrix": str(origin_transition_matrix_path),
    "origin_audit_table": str(origin_audit_table_path),
    "final_variable_dictionary": str(final_variable_dictionary_path),
}

row_counts = {
    "source_message_rows_loaded": int(source_file_summary["rows"].sum()),
    "messages_after_basic_cleaning": int(len(messages)),
    "provider_master_clean_rows": int(len(provider_master_clean)),
    "messages_after_provider_enrichment": int(len(messages_provider_enriched)),
    "messages_after_sender_receiver_refinement": int(len(messages_refined)),
    "messages_without_threads": int(len(messages_without_threads)),
}

input_file_names = {
    "cleaned_message_files": sorted(source_file_summary["source_file"].astype(str).tolist()),
    "provider_source_files": sorted([
        provider_master_source_path.name,
        unmapped_users_path.name,
        missing_users_path.name,
    ]),
}

reproducibility_summary = {
    "run_id": RUN_ID,
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": package_version("pyarrow"),
    },
    "input_file_names": input_file_names,
    "key_output_files": key_output_files,
    "row_counts": row_counts,
    "downstream_handoff": {
        "handoff_dataset": "messages_without_threads.parquet",
        "threading_notebook": "02_threading_diagnostics_sensitivity.ipynb",
        "demographics_cci_notebook": "04_build_demographics_cci_clean.ipynb",
    },
    "privacy_restriction": "Row-level output datasets are restricted EHR-derived records and should not be publicly shared. Share code and aggregate QA summaries only.",
}

reproducibility_summary_path = RUN_DIR / "qa_reproducibility_summary.json"
with open(reproducibility_summary_path, "w", encoding="utf-8") as f:
    json.dump(reproducibility_summary, f, indent=2, default=str)

print("Saved reproducibility summary:")
print(" -", reproducibility_summary_path)
print(json.dumps({"row_counts": row_counts, "package_versions": reproducibility_summary["package_versions"]}, indent=2))


## Expected Outputs

Every run writes to a unique folder under `runs/core_message_preprocessing_YYYYMMDD_HHMMSS/`.

Core outputs:

- `messages_core_preprocessed.parquet`: stacked and cleaned message table before provider mapping, sender/source classification, threading, demographics/CCI, and visits.
- `message_source_file_summary.csv`: row counts by source message file.
- `message_type_summary.csv`: row counts by message type.
- `subject_clean_summary_top100.csv`: aggregate subject-cleaning QA only.
- `qa_core_message_preprocessing_summary.json`: run metadata and QA checks.
- `provider_master_clean.parquet`: governed provider master used for message enrichment.
- `messages_provider_enriched.parquet`: message table enriched with sender/receiver provider metadata before final sender/source classification.
- `provider_type_bucket_counts.csv`: provider role-bucket QA.
- `sender_category_counts_initial.csv`: initial sender category QA after provider merge.
- `receiver_category_counts_initial.csv`: initial receiver category QA after provider merge.
- `qa_provider_enrichment_summary.json`: provider-master and enrichment QA checks.
- `messages_without_threads.parquet`: final pre-thread message-level dataset with `Sender_all`, `Receiver_all`, `origin`, and audit variables. This is the handoff file for threading and message-time CCI.
- `sender_all_counts.csv`: final analytic sender-category QA.
- `receiver_all_counts.csv`: final analytic receiver-category QA.
- `origin_original_counts.csv`: aggregate origin QA before final system-identification rules.
- `origin_counts.csv`: aggregate System/Patient/Provider final origin QA.
- `origin_transition_matrix.csv`: transition table comparing original structured origin with final analytic origin.
- `origin_audit_table.csv`: concise origin-change QA table, including Patient-origin stability.
- `qa_sender_source_classification_summary.json`: sender/source classification QA checks.
- `final_variable_dictionary.csv`: data dictionary for final variables created in this notebook.
- `qa_reproducibility_summary.json`: runtime, package-version, input/output, privacy, row-count, and downstream-handoff summary.

Thread-derived outputs are created by `02_threading_diagnostics_sensitivity.ipynb`, not by this preprocessing notebook.
